In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.3


In [ ]:
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw


,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


1. Profile the data — find the problems

1A. A FEW COMMANDS REVEAL MOST PROBLEMS

In [ ]:


print('Missing per column:')
print(raw.isna().sum())

print('\nDuplicate rows:', raw.duplicated().sum())

print('\nNote: spend is type', raw['spend'].dtype, '-> stored as text!')

print('\nData types:')
print(raw.dtypes)


Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!

Data types:
id         int64
name      object
age      float64
city      object
spend     object
date      object
dtype: object


LAB EXERCISE 1

 1. duplicate row count

In [ ]:
print(raw.duplicated().sum())


1


2. missing per column

In [ ]:
print(raw.isna().sum())


id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


3. Problems I can see

In [ ]:
print("Problems: duplicate row, missing age, invalid age (-1), inconsistent city names, spend stored as text, N/A in spend")


Problems: duplicate row, missing age, invalid age (-1), inconsistent city names, spend stored as text, N/A in spend


Missing values — detect & handle

2A. UNMASK DISGUISED MISSING VALUES

In [ ]:
raw['spend'] = pd.to_numeric(raw['spend'], errors='coerce')
raw['age'] = raw['age'].replace(-1, np.nan)

print('Missing after unmasking:')
print(raw[['age', 'spend']].isna().sum())


Missing after unmasking:
age      2
spend    1
dtype: int64


2B. HANDLE THE GAPS (impute)

In [ ]:
raw['age'] = raw['age'].fillna(raw['age'].median())
raw['spend'] = raw['spend'].fillna(raw['spend'].median())
print('Missing after imputing:', raw[['age', 'spend']].isna().sum().sum())


Missing after imputing: 0


LAB EXERCISE 2 — Compare drop vs impute


In [ ]:
ex = raw.copy()


1. unmask missing values (spend -> numeric, age -1 -> NaN)

In [ ]:
ex = raw.copy()
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)


2a. dropna version

In [ ]:
ex = raw.copy()
dropna_version = ex.dropna()


2b. median-impute version

In [ ]:

impute_version = ex.copy()
impute_version['age'] = impute_version['age'].fillna(impute_version['age'].median())
impute_version['spend'] = impute_version['spend'].fillna(impute_version['spend'].median())


3. compare row counts

In [ ]:
print(len(dropna_version))
print(len(impute_version))


8
8


3. Duplicates & data types

3A. DROP DUPLICATE ROWS

In [ ]:
print('Before:', raw.shape)
raw = raw.drop_duplicates()
print('After :', raw.shape, '-> removed the repeated Gus row')


Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


3B. FIX DATA TYPES

In [ ]:
raw['date'] = pd.to_datetime(raw['date'])

raw['city'] = raw['city'].astype('string')
print(raw.dtypes)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


/tmp/ipykernel_1735/1696751417.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['date'] = pd.to_datetime(raw['date'])
/tmp/ipykernel_1735/1696751417.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['city'] = raw['city'].astype('string')


LAB EXERCISE 3 — Dedupe & retype

In [ ]:
ex = raw.copy()


1. fix types: spend -> numeric, date -> datetime

In [ ]:
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'])


2. drop duplicates

In [ ]:
ex = ex.drop_duplicates()


3. dtypes + shape

In [ ]:
print(ex.dtypes)
print(ex.shape)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object
(7, 6)


4. Outliers — detect with the IQR rule

4A. THE IQR RULE

In [ ]:
q1, q3 = raw['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = raw[(raw['spend'] < low) | (raw['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])


Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


4B. ONE WAY TO TREAT THEM — CAP (winsorise)

In [ ]:
raw['spend_capped'] = raw['spend'].clip(lower=low, upper=high)
print(raw[['name', 'spend', 'spend_capped']])


  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


/tmp/ipykernel_1735/139548929.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['spend_capped'] = raw['spend'].clip(lower=low, upper=high)


 LAB EXERCISE 4 — Find the outliers


1. Q1, Q3, IQR for 'age'

In [ ]:
q1, q3 = raw['age'].quantile([0.25, 0.75])
iqr = q3 - q1


 2. lower & upper bounds

In [ ]:
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr


3. rows outside the bounds

In [ ]:
outliers = raw[(raw['age'] < low) | (raw['age'] > high)]
print(outliers)


Empty DataFrame
Columns: [id, name, age, city, spend, date, spend_capped]
Index: []


5. Messy text & inconsistent categories

5A. THE PROBLEM — ONE CITY, MANY SPELLINGS

In [ ]:
print(raw['city'].value_counts())


city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


 5B. STANDARDISE THE STRINGS

In [ ]:
s = raw['city'].astype('string')
s = s.str.strip()
s = s.str.lower()
s = s.str.replace('.', '', regex=False)
s = s.replace({'bombay': 'mumbai'})
raw['city'] = s
print(raw['city'].value_counts())


city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


/tmp/ipykernel_1735/2257874432.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['city'] = s


LAB EXERCISE 5 — Clean a messy column


In [ ]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')


1. strip + lower

In [ ]:
messy = messy.str.strip().str.lower()


2. Map 'n.y.' → 'new york'


In [ ]:
messy = messy.replace({'n.y.': 'new york'})


3. Value counts

In [ ]:
print(messy.value_counts())


london      3
new york    3
Name: count, dtype: Int64


The cleaned dataset

In [ ]:
clean = raw.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean


Final shape: (7, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
